# What structure is hidden in the market? Clusters and regimes

**Question.** Without giving a model any labels, what structure can it find in market data? Two unsupervised learning tasks:

1. **Market map:** group about 40 large US and Hong Kong stocks by how they move. Do the groups follow *country* or *industry*?
2. **Market regimes:** does the market switch between hidden calm and stormy phases, and how long do they last?

**Methods**
- Weekly returns, which avoid the HK/US time-zone mismatch described in notebook 01.
- **PCA** on the correlation matrix to find the main patterns of co-movement.
- **Hierarchical clustering** (average linkage) on correlation distance √(2(1 − ρ)), with the number of groups chosen by **silhouette score** (searched from 2 to 5 for readability).
- **Adjusted Rand index (ARI)** to compare the groups with country and sector labels. 1 means a perfect match and about 0 means random.
- A two-state **Gaussian Hidden Markov Model**, fitted with the **Baum–Welch (EM)** algorithm. I implemented it from scratch in `backend/app/unsupervised.py`, and it is unit-tested against simulated regimes and a brute-force likelihood.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram

sys.path.insert(0, str(Path("..") / "backend"))
import datavest_style  # site colours and fonts
from datavest_style import SERIES, INK, CINNABAR
from app import unsupervised as un
from app.config import RESEARCH_UNIVERSE, WATCHLIST_GROUPS

NAMES = {s: n for g in WATCHLIST_GROUPS.values() for s, n in g.items()} | {s: v[0] for s, v in RESEARCH_UNIVERSE.items()}
pd.options.display.float_format = "{:,.3f}".format

with sqlite3.connect(Path("..") / "backend" / "datavest.db") as con:
    long = pd.read_sql("SELECT t.symbol, p.date, p.adj_close, p.close FROM prices p JOIN tickers t ON t.id = p.ticker_id",
                       con, parse_dates=["date"])
long["adj_close"] = long["adj_close"].fillna(long["close"])
prices = long.pivot(index="date", columns="symbol", values="adj_close").sort_index()
stocks = [s for s in RESEARCH_UNIVERSE if s in prices.columns]
print(f"{len(stocks)} stocks in the research universe, {prices.index[0].date()} to {prices.index[-1].date()}")

## 1. Patterns of co-movement (PCA)

In [ ]:
w = un.weekly_return_matrix(prices[stocks])
p = un.pca(w, n=5)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar([f"PC{i}" for i in range(1, 6)], p["explained"], color=SERIES[0], width=0.6)
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.set_title("Share of all weekly movement explained by each pattern");

In [ ]:
load = p["loadings"].copy()
load["region"] = ["Hong Kong" if s.endswith(".HK") else "United States" for s in load.index]
load.groupby("region")[["PC1", "PC2", "PC3"]].mean().rename_axis(None).style.format("{:+.2f}").set_caption(
    "Average loading by region: PC1 is the whole market; a sign split on PC2 means it separates the regions")

## 2. Clustering

In [ ]:
res = un.cluster(w)
sil = pd.Series(res["silhouette"], name="silhouette score")
ax = sil.plot(marker="o", color=SERIES[0], figsize=(7, 3.2), title="Silhouette score by number of groups (higher is better)")
ax.set_xlabel("number of groups"); ax.set_xticks(list(sil.index));
print(f"chosen: {res['k']} groups")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
dendrogram(res["linkage"], labels=[NAMES.get(s, s) for s in w.columns], orientation="left", ax=ax,
           color_threshold=None, above_threshold_color=INK, leaf_font_size=9)
ax.set_title("How the stocks join into groups (shorter branches = move more alike)")
ax.set_xlabel("correlation distance"); ax.grid(axis="y", visible=False);

In [ ]:
labels = res["labels"]
sector = pd.Series({s: RESEARCH_UNIVERSE[s][1] for s in w.columns})
region = pd.Series({s: "Hong Kong" if s.endswith(".HK") else "United States" for s in w.columns})
table = pd.DataFrame({"group": labels, "region": region, "sector": sector, "name": [NAMES[s] for s in w.columns]})
summary = table.groupby("group").agg(stocks=("name", "count"),
                                     regions=("region", lambda x: ", ".join(f"{k} {v}" for k, v in x.value_counts().items())),
                                     main_sectors=("sector", lambda x: ", ".join(x.value_counts().index[:3])))
summary.rename_axis(None)

In [ ]:
ari = pd.Series({"country": un.agreement(labels, region), "sector": un.agreement(labels, sector)}, name="adjusted Rand index")
ari.to_frame().style.format("{:.2f}").set_caption("How well the groups match each label (1 = perfectly, 0 = random)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))
for g in sorted(labels.unique()):
    for reg, marker in [("United States", "o"), ("Hong Kong", "D")]:
        idx = [s for s in w.columns if labels[s] == g and region[s] == reg]
        ax.scatter(p["loadings"].loc[idx, "PC2"], p["loadings"].loc[idx, "PC3"], s=60, marker=marker,
                   color=SERIES[(g - 1) % len(SERIES)], edgecolor="white", linewidth=0.8,
                   label=f"group {g}, {reg}" if idx else None)
for s in w.columns:
    ax.annotate(NAMES[s], (p["loadings"].loc[s, "PC2"], p["loadings"].loc[s, "PC3"]), fontsize=7.5, xytext=(4, 3), textcoords="offset points", color=INK)
ax.set_xlabel(f"pattern 2 ({p['explained'][1]:.0%})"); ax.set_ylabel(f"pattern 3 ({p['explained'][2]:.0%})")
ax.set_title("Market map: stocks close together move together"); ax.legend(fontsize=8, loc="best");

## 3. Market regimes (Hidden Markov Model)

In [ ]:
rows, results = {}, {}
for sym in ["^GSPC", "^HSI"]:
    r = un.regimes(prices[sym].dropna())
    results[sym] = r
    for i, name in enumerate(["calm", "stormy"]):
        rows[(NAMES[sym], name)] = {"yearly volatility": r["vol"][i], "average spell (days)": r["expected_days"][i],
                                    "share of time": r["share_of_time"][i], "average daily return": r["model"].mean[i]}
pd.DataFrame(rows).T.style.format({"yearly volatility": "{:.0%}", "average spell (days)": "{:.0f}",
                                   "share of time": "{:.0%}", "average daily return": "{:+.3%}"})

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for ax, sym in zip(axes, ["^GSPC", "^HSI"]):
    close = prices[sym].dropna()
    pt = results[sym]["p_turbulent_smoothed"].reindex(close.index).fillna(0)
    ax.plot(close.index, close, color=INK, lw=1.2)
    ax.fill_between(close.index, close.min(), close.max(), where=pt > 0.5, color=CINNABAR, alpha=0.15, step="mid", label="stormy phase")
    ax.set_title(f"{NAMES[sym]}: stormy phases found by the HMM"); ax.legend(loc="upper left")
fig.tight_layout()

**Filtered vs smoothed.** The shading uses *smoothed* probabilities, which use the whole sample and are good for describing history. The site's "today" reading uses *filtered* probabilities, which use only data up to today. That is the only honest way to say which regime we are in now.

## 4. Findings

In [ ]:
lines = [
    f"PCA: the first pattern (the whole market moving together) explains {p['explained'][0]:.0%} of weekly movement across {len(w.columns)} stocks.",
    f"Clustering chose **{res['k']} groups**. They match **country** with ARI {ari['country']:.2f} and **sector** with ARI {ari['sector']:.2f}: "
    + ("neither label explains the groups well." if max(ari) < 0.1
       else "where a stock is listed matters more than what the company does." if ari["country"] > ari["sector"]
       else "industry matters more than where a stock is listed."),
]
for sym in ["^GSPC", "^HSI"]:
    r = results[sym]
    lines.append(f"{NAMES[sym]}: stormy phases were {r['vol'][1] / r['vol'][0]:.1f}× as volatile as calm ones, lasted about {r['expected_days'][1]:.0f} trading days on average, and covered {r['share_of_time'][1]:.0%} of the period. "
                 f"Today: {r['p_turbulent_filtered'].iloc[-1]:.0%} probability of the stormy phase.")
from IPython.display import Markdown
Markdown("\n\n".join("- " + l for l in lines))

## Limitations

- **Survivorship:** the universe is today's large companies, so stocks that shrank or disappeared are missing.
- **Clusters depend on the method.** Average linkage on correlation is one reasonable choice. Ward linkage or k-means on PCA scores can split groups differently. The silhouette score guides the number of groups but doesn't prove it.
- **Two regimes is a simplification.** Three states (calm / normal / crisis) or Student-t emissions may fit better. Model choice by BIC would be a natural extension.
- **Regimes are not forecasts.** A stormy phase can end tomorrow. The model's switching probabilities describe averages, not the next move.